In [ ]:
# Spatial Domain ResNet50 with Comprehensive Metrics
# Computes: Accuracy, Cohen's Kappa, Precision, Recall, F1, Specificity, Error Rate

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    cohen_kappa_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
import os
import warnings
import gc
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

# ================== Step 1: Custom Dataset for Fruits-360 ==================

class FruitsDataset(Dataset):
    """Custom dataset for Fruits-360"""

    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted([d for d in os.listdir(root_dir)
                               if os.path.isdir(os.path.join(root_dir, d))])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}

        self.samples = []
        for class_name in self.classes:
            class_dir = os.path.join(root_dir, class_name)
            if os.path.isdir(class_dir):
                for img_name in os.listdir(class_dir):
                    if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                        self.samples.append((
                            os.path.join(class_dir, img_name),
                            self.class_to_idx[class_name]
                        ))

        print(f"Found {len(self.samples)} images in {len(self.classes)} classes")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            return torch.zeros(3, 100, 100), label


def load_fruits_dataset(data_root):
    """Load Fruits-360 dataset with augmentation"""

    # Training transform with augmentation
    transform_train = transforms.Compose([
        transforms.Resize((100, 100)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.RandomRotation(20),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Test/Val transform — no augmentation
    transform_test = transforms.Compose([
        transforms.Resize((100, 100)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    train_dir = os.path.join(data_root, 'Training')
    test_dir  = os.path.join(data_root, 'Test')

    trainset = FruitsDataset(train_dir, transform=transform_train)
    testset  = FruitsDataset(test_dir,  transform=transform_test)

    return trainset, testset, trainset.classes

# ================== Step 2: Spatial Domain ResNet50 Model ==================

class SpatialResNet50(nn.Module):
    """
    Standard ResNet50 fine-tuned for Fruits-360.
    Operates entirely in the spatial domain — no FFT involved.
    """

    def __init__(self, num_classes, dropout_rate=0.5):
        super(SpatialResNet50, self).__init__()

        # Load pretrained ResNet50
        self.backbone = models.resnet50(pretrained=True)

        # Freeze early layers (layer1 and layer2) to retain general features
        layers_to_freeze = [
            self.backbone.conv1,
            self.backbone.bn1,
            self.backbone.layer1,
            self.backbone.layer2,
        ]
        for layer in layers_to_freeze:
            for param in layer.parameters():
                param.requires_grad = False

        # Number of features before the original FC layer
        num_features = self.backbone.fc.in_features  # 2048 for ResNet50

        # Replace the original FC with a custom classifier head
        self.backbone.fc = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 1024),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(1024),
            nn.Dropout(dropout_rate * 0.7),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(512, num_classes)
        )

        self._initialize_head_weights()

    def _initialize_head_weights(self):
        """Initialize the custom classifier head."""
        for m in self.backbone.fc.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        return self.backbone(x)

# ================== Step 3: Early Stopping ==================

class EarlyStopping:
    """Early stopping to prevent overfitting."""

    def __init__(self, patience=10, min_delta=0.0, verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_accuracy, model):
        score = val_accuracy

        if self.best_score is None:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif score < self.best_score + self.min_delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0

# ================== Step 4: Training Function ==================

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001, weight_decay=1e-4):
    """Train the spatial ResNet50 model and return history."""

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    # Separate frozen and unfrozen parameters
    trainable_params   = [p for p in model.parameters() if p.requires_grad]
    untrainable_params = [p for p in model.parameters() if not p.requires_grad]

    print(f"Trainable parameters   : {sum(p.numel() for p in trainable_params):,}")
    print(f"Frozen parameters      : {sum(p.numel() for p in untrainable_params):,}")

    optimizer = torch.optim.AdamW(trainable_params, lr=lr, weight_decay=weight_decay)

    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=10, T_mult=2, eta_min=1e-7
    )

    early_stopping = EarlyStopping(patience=15, min_delta=0.1, verbose=True)

    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    best_val_accuracy = 0.0
    best_model_state  = None

    # Mixed precision scaler (GPU only)
    scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

    for epoch in range(epochs):

        # ---- Training phase ----
        model.train()
        running_loss  = 0.0
        correct_train = 0
        total_train   = 0

        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]", leave=False)
        for i, (images, labels) in enumerate(train_pbar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            if scaler is not None:
                with torch.cuda.amp.autocast():
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss at batch {i}, skipping...")
                    continue

                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)

                if torch.isnan(loss) or torch.isinf(loss):
                    print(f"Warning: NaN/Inf loss at batch {i}, skipping...")
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            running_loss  += loss.item()
            _, predicted   = torch.max(outputs.data, 1)
            total_train   += labels.size(0)
            correct_train += (predicted == labels).sum().item()

            train_pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc' : f'{100 * correct_train / total_train:.2f}%'
            })

            # Periodic GPU cache clearing
            if i % 50 == 0 and torch.cuda.is_available():
                torch.cuda.empty_cache()

        scheduler.step()

        avg_train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct_train / total_train
        train_losses.append(avg_train_loss)
        train_accuracies.append(train_accuracy)

        # ---- Validation phase ----
        model.eval()
        running_val_loss = 0.0
        correct          = 0
        total            = 0

        with torch.no_grad():
            val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]", leave=False)
            for images, labels in val_pbar:
                images = images.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)

                if scaler is not None:
                    with torch.cuda.amp.autocast():
                        outputs = model(images)
                        loss    = criterion(outputs, labels)
                else:
                    outputs = model(images)
                    loss    = criterion(outputs, labels)

                running_val_loss += loss.item()
                _, predicted      = torch.max(outputs.data, 1)
                total            += labels.size(0)
                correct          += (predicted == labels).sum().item()

                val_pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'acc' : f'{100 * correct / total:.2f}%'
                })

        avg_val_loss = running_val_loss / len(val_loader)
        val_accuracy = 100 * correct / total
        val_losses.append(avg_val_loss)
        val_accuracies.append(val_accuracy)

        # Save best model
        if val_accuracy > best_val_accuracy:
            best_val_accuracy = val_accuracy
            best_model_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        print(f'\nEpoch [{epoch+1}/{epochs}]')
        print(f'  Train Loss: {avg_train_loss:.4f} | Train Acc: {train_accuracy:.2f}%')
        print(f'  Val   Loss: {avg_val_loss:.4f} | Val   Acc: {val_accuracy:.2f}%')
        print(f'  LR        : {optimizer.param_groups[0]["lr"]:.8f}')

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        early_stopping(val_accuracy, model)
        if early_stopping.early_stop:
            print("Early stopping triggered!")
            break

    # Restore best weights
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        print(f"\nRestored best model — Val Accuracy: {best_val_accuracy:.2f}%")

    return train_losses, val_losses, train_accuracies, val_accuracies

# ================== Step 5: Comprehensive Metrics Computation ==================

def compute_specificity_multiclass(y_true, y_pred, num_classes):
    """
    Compute macro-averaged specificity for a multi-class problem.

    For each class c:
        TN_c = samples correctly NOT predicted as c
        FP_c = samples incorrectly predicted as c
        Specificity_c = TN_c / (TN_c + FP_c)

    Returns the macro average across all classes.
    """
    specificities = []
    for c in range(num_classes):
        # One-vs-rest binary mask
        true_binary = (np.array(y_true) == c).astype(int)
        pred_binary = (np.array(y_pred) == c).astype(int)

        tn = np.sum((true_binary == 0) & (pred_binary == 0))
        fp = np.sum((true_binary == 0) & (pred_binary == 1))

        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        specificities.append(spec)

    return float(np.mean(specificities))


def evaluate_model(model, data_loader, num_classes, split_name="Test"):
    """
    Evaluate the model on a data loader and compute all metrics.

    Returns a dictionary with:
        - accuracy
        - cohen_kappa
        - precision (macro)
        - recall    (macro)
        - f1        (macro)
        - specificity (macro)
        - error_rate
        - all_predictions
        - all_labels
    """
    model.eval()
    all_predictions = []
    all_labels      = []
    correct         = 0
    total           = 0

    with torch.no_grad():
        pbar = tqdm(data_loader, desc=f"Evaluating [{split_name}]")
        for images, labels in pbar:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs   = model(images)
            _, predicted = torch.max(outputs.data, 1)

            total   += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({'acc': f'{100 * correct / total:.2f}%'})

    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)

    # ---- Core metrics ----
    accuracy     = 100.0 * correct / total
    error_rate   = 100.0 - accuracy
    kappa        = cohen_kappa_score(y_true, y_pred)
    precision    = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall       = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1           = f1_score(y_true, y_pred, average='macro', zero_division=0)
    specificity  = compute_specificity_multiclass(y_true, y_pred, num_classes)

    metrics = {
        'accuracy'   : accuracy,
        'error_rate' : error_rate,
        'cohen_kappa': kappa,
        'precision'  : precision,
        'recall'     : recall,
        'f1'         : f1,
        'specificity': specificity,
        'predictions': y_pred,
        'labels'     : y_true,
    }

    return metrics


def print_metrics(metrics, split_name="Test"):
    """Pretty-print all computed metrics."""
    print("\n" + "=" * 60)
    print(f"  COMPREHENSIVE METRICS — {split_name.upper()} SET")
    print("=" * 60)
    print(f"  Overall Accuracy      : {metrics['accuracy']:.4f} %")
    print(f"  Overall Error Rate    : {metrics['error_rate']:.4f} %")
    print(f"  Cohen's Kappa Score   : {metrics['cohen_kappa']:.4f}")
    print(f"  Overall Precision     : {metrics['precision']:.4f}  (macro avg)")
    print(f"  Overall Recall        : {metrics['recall']:.4f}  (macro avg)")
    print(f"  Overall F1 Score      : {metrics['f1']:.4f}  (macro avg)")
    print(f"  Overall Specificity   : {metrics['specificity']:.4f}  (macro avg)")
    print("=" * 60)

# ================== Step 6: Training Curve Plotting ==================

def plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies):
    """Plot training and validation loss / accuracy curves."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(train_losses) + 1)

    ax1.plot(epochs, train_losses, 'b-', label='Training Loss',   linewidth=2)
    ax1.plot(epochs, val_losses,   'r-', label='Validation Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss',  fontsize=12)
    ax1.set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, train_accuracies, 'b-', label='Training Accuracy',   linewidth=2)
    ax2.plot(epochs, val_accuracies,   'r-', label='Validation Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch',        fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Training curves saved to 'training_curves.png'")


def plot_metrics_bar(metrics, split_name="Test"):
    """Bar chart of the seven key metrics (all normalised to [0, 1] or %)."""
    labels = [
        'Accuracy (%)',
        'Error Rate (%)',
        "Cohen's Kappa",
        'Precision',
        'Recall',
        'F1 Score',
        'Specificity',
    ]
    values = [
        metrics['accuracy'],
        metrics['error_rate'],
        metrics['cohen_kappa'] * 100,   # scale to % for visual comparison
        metrics['precision'] * 100,
        metrics['recall'] * 100,
        metrics['f1'] * 100,
        metrics['specificity'] * 100,
    ]
    colors = ['#2196F3', '#F44336', '#9C27B0', '#4CAF50', '#FF9800', '#00BCD4', '#795548']

    fig, ax = plt.subplots(figsize=(12, 6))
    bars = ax.bar(labels, values, color=colors, edgecolor='black', alpha=0.85)

    for bar, val in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2.0,
            bar.get_height() + 0.5,
            f'{val:.2f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold'
        )

    ax.set_ylim(0, 115)
    ax.set_ylabel('Score (% or ×100 for [0,1] metrics)', fontsize=11)
    ax.set_title(f'Comprehensive Model Metrics — {split_name} Set\n(ResNet50, Spatial Domain)',
                 fontsize=13, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig('metrics_bar_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Metrics bar chart saved to 'metrics_bar_chart.png'")

# ================== Step 7: Main Execution Pipeline ==================

def main():
    print("=" * 80)
    print("  Spatial Domain ResNet50 — Fruits-360 Classification")
    print("  Metrics: Accuracy | Kappa | Precision | Recall | F1 | Specificity | Error Rate")
    print("=" * 80)

    # ---- Dataset path — MODIFY IF NEEDED ----
    data_root = r'C:\Users\CSE_SDPL\Downloads\fruits-360_100x100\fruits-360'

    if not os.path.exists(data_root):
        print(f"\nERROR: Dataset path not found: {data_root}")
        print("Please set 'data_root' to the correct location.")
        return

    # ---- Step 1: Load dataset ----
    print("\n[Step 1] Loading Fruits-360 dataset...")
    try:
        trainset, testset, classes = load_fruits_dataset(data_root)
        num_classes = len(classes)
        print(f"Number of classes: {num_classes}")
    except Exception as e:
        print(f"ERROR loading dataset: {e}")
        return

    # ---- Step 2: Train / Val split ----
    print("\n[Step 2] Splitting training set into train / validation (85 / 15)...")
    train_size = int(0.85 * len(trainset))
    val_size   = len(trainset) - train_size
    train_subset, val_subset = torch.utils.data.random_split(
        trainset, [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )
    print(f"  Training   samples : {len(train_subset)}")
    print(f"  Validation samples : {len(val_subset)}")
    print(f"  Test       samples : {len(testset)}")

    # ---- Step 3: Data loaders ----
    print("\n[Step 3] Creating data loaders...")
    batch_size  = 128
    num_workers = 4 if os.name != 'nt' else 0
    print(f"  Batch size  : {batch_size}")
    print(f"  Num workers : {num_workers}")

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None
    )
    # Use larger batch for test (no gradients needed)
    test_loader = DataLoader(
        testset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
        prefetch_factor=2 if num_workers > 0 else None
    )

    # ---- Step 4: Build model ----
    print("\n[Step 4] Initialising ResNet50 (spatial domain)...")
    model = SpatialResNet50(num_classes=num_classes, dropout_rate=0.5).to(device)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total parameters : {total_params:,}")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"  GPU memory allocated : {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

    # ---- Step 5: Train ----
    print("\n[Step 5] Training model...")
    try:
        train_losses, val_losses, train_accuracies, val_accuracies = train_model(
            model, train_loader, val_loader,
            epochs=50,
            lr=0.001,
            weight_decay=5e-4
        )
    except Exception as e:
        import traceback
        print(f"\nERROR during training: {e}")
        traceback.print_exc()
        return

    # ---- Step 5.1: Plot training curves ----
    print("\n[Step 5.1] Plotting training curves...")
    plot_training_curves(train_losses, val_losses, train_accuracies, val_accuracies)

    # ---- Step 6: Evaluate on test set ----
    print("\n[Step 6] Evaluating on the test set...")
    test_metrics = evaluate_model(model, test_loader, num_classes, split_name="Test")
    print_metrics(test_metrics, split_name="Test")

    # ---- Step 7: Plot metrics ----
    print("\n[Step 7] Plotting metrics bar chart...")
    plot_metrics_bar(test_metrics, split_name="Test")

    # ---- Step 8: Per-class classification report ----
    print("\n[Step 8] Per-class classification report (first 20 classes shown)...")
    report = classification_report(
        test_metrics['labels'],
        test_metrics['predictions'],
        target_names=classes,
        zero_division=0
    )
    # Print only first 20 classes to avoid wall of text
    report_lines = report.split('\n')
    # Header + up to 20 class rows + footer
    visible_lines = report_lines[:2] + report_lines[2:22] + ['    ...'] + report_lines[-4:]
    print('\n'.join(visible_lines))

    # ---- Step 9: Save model ----
    print("\n[Step 9] Saving trained model...")
    try:
        torch.save({
            'model_state_dict' : model.state_dict(),
            'test_accuracy'    : test_metrics['accuracy'],
            'test_kappa'       : test_metrics['cohen_kappa'],
            'test_precision'   : test_metrics['precision'],
            'test_recall'      : test_metrics['recall'],
            'test_f1'          : test_metrics['f1'],
            'test_specificity' : test_metrics['specificity'],
            'test_error_rate'  : test_metrics['error_rate'],
            'classes'          : classes,
            'num_classes'      : num_classes,
        }, 'fruits_resnet50_spatial.pth')
        print("  Model saved as 'fruits_resnet50_spatial.pth'")
    except Exception as e:
        print(f"  ERROR saving model: {e}")

    # ---- Final summary ----
    print("\n" + "=" * 80)
    print("  FINAL SUMMARY")
    print("=" * 80)
    print_metrics(test_metrics, split_name="Test")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"Final GPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

    print("\nPipeline completed successfully!")

if __name__ == "__main__":
    main()